# CV models
Reusable code lives in `cv/telemetry_ocr.py` and `cv/video_embedding.py`. There are no ground-truth labels yet, so the temporal model is self-supervised and its numbers are **not accuracy**.

In [1]:
# %pip install opencv-python pytesseract torch torchvision
import os, sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'cv' / 'flame_cv.py').exists())
sys.path.insert(0, str(ROOT))
DATA = Path(os.environ.get('NASA_DATA_DIR', ROOT / 'datasets' / 'samples' / 'cv'))
VIDEO = DATA / 'FF-PSI-25' / 'videos' / 'PSI-25_Raw Data_BASS_GMT50_SAMPLE_147_TEST_B1.mp4'
IMAGES = DATA / 'FF-PSI-107' / 'SPICE_GMT045_2009_jpegs'
# Set TESSERACT_CMD if tesseract is not on PATH, e.g. C:\Program Files\Tesseract-OCR\tesseract.exe

## 1. Telemetry OCR (per-field ROI, validation, multi-frame voting)
Calibrate `rois` once from a frame; hand-label ~30-50 frames and use `field_accuracy` to measure exact match.

In [5]:
from cv.telemetry_ocr import OCRConfig, extract_from_video, field_accuracy
import shutil

tesseract = os.environ.get("TESSERACT_CMD") or shutil.which("tesseract")
if not tesseract:
    default_tesseract = Path(r"C:\Program Files\Tesseract-OCR\tesseract.exe")
    if default_tesseract.is_file():
        tesseract = str(default_tesseract)
if not tesseract:
    raise FileNotFoundError(
        "Tesseract OCR is not installed. Install it and set TESSERACT_CMD to its executable path."
    )

cfg = OCRConfig(tesseract_cmd=tesseract)  # set cfg.rois = {'fuel': (x0,y0,x1,y1), ...} after calibration
result = extract_from_video(str(VIDEO), cfg)
result

{'fuel': {'value': None, 'confidence': 0.0, 'votes': 0},
 'air': {'value': None, 'confidence': 0.0, 'votes': 0},
 'fan': {'value': None, 'confidence': 0.0, 'votes': 0},
 'rad': {'value': None, 'confidence': 0.0, 'votes': 0}}

## 2. Label-free temporal model on frozen ResNet embeddings (GPU/AMP if available)

In [2]:
import cv2
from cv.video_embedding import list_images, extract_embeddings, train_next_embedding, get_device
print('device:', get_device())
frames = [cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB) for p in list_images(str(IMAGES))]
emb = extract_embeddings(frames)
res = train_next_embedding(emb, epochs=200)
{k: v for k, v in res.items() if k != 'model'}

device: cpu
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\HP/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100.0%


{'val_mse': 0.9590643048286438,
 'copy_previous_mse': 2.1309101581573486,
 'beats_baseline': True,
 'note': 'self-supervised; not an accuracy metric'}